In [45]:
import pandas as pd
import numpy as np

In [46]:
df_sales= pd.read_csv(r"C:\Users\ahmed\Documents\Data Science LUXDEV\PPORTFOLIO PROJECTS\INVENTORY FORECASTING\CSV\CLEAN\feature_engineered_sales.csv")
df_sales.head()
df_sales.shape
df_sales['date']=pd.to_datetime(df_sales['date'])
df_sales = df_sales.sort_values(['product_id','date'])

In [47]:
df_sales.head()

,product_id,product_name_x,category_x,units_sold,unit_price_x,promotion_flag,stock_level_end_of_day,stockout_flag,date,product_name_y,...,Month,Quarter,is_festive,day_of_year,Revenue,rolling_7d_avg,rolling_28d_avg,previous_day_sale,previous_week_sale,days_since_last_promo
0,SKU001,Soda 500ml,Beverages,29,154.61,False,365.9,False,2024-01-01,Soda 500ml,...,1,1,False,1,4483.69,29.0,29.0,NaN,NaN,0
1,SKU001,Soda 500ml,Beverages,27,154.61,False,338.9,False,2024-01-02,Soda 500ml,...,1,1,False,2,4174.47,28.0,28.0,29.0,NaN,0
2,SKU001,Soda 500ml,Beverages,25,154.61,False,313.9,False,2024-01-03,Soda 500ml,...,1,1,False,3,3865.25,27.0,27.0,27.0,NaN,0
3,SKU001,Soda 500ml,Beverages,33,154.61,True,280.9,False,2024-01-04,Soda 500ml,...,1,1,False,4,5102.13,28.5,28.5,25.0,NaN,0
4,SKU001,Soda 500ml,Beverages,26,154.61,False,254.9,False,2024-01-05,Soda 500ml,...,1,1,False,5,4019.86,28.0,28.0,33.0,NaN,0


In [48]:
df_sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18275 entries, 0 to 18274
Data columns (total 27 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   product_id              18275 non-null  object        
 1   product_name_x          18275 non-null  object        
 2   category_x              18275 non-null  object        
 3   units_sold              18275 non-null  int64         
 4   unit_price_x            18275 non-null  float64       
 5   promotion_flag          18275 non-null  bool          
 6   stock_level_end_of_day  18275 non-null  float64       
 7   stockout_flag           18275 non-null  bool          
 8   date                    18275 non-null  datetime64[ns]
 9   product_name_y          18275 non-null  object        
 10  category_y              18275 non-null  object        
 11  unit_cost               18275 non-null  float64       
 12  unit_price_y            18275 non-null  float6

In [49]:
df_product_master = pd.read_csv(r"C:\Users\ahmed\Documents\Data Science LUXDEV\PPORTFOLIO PROJECTS\INVENTORY FORECASTING\CSV\CLEAN\products_cleaned.csv")
df_product_master.head()
df_products =df_product_master[['product_id','product_name','category','lead_time_days','safety_stock_days']]
df_products.head()

,product_id,product_name,category,lead_time_days,safety_stock_days
0,SKU001,Soda 500ml,Beverages,5,4
1,SKU002,Bottled Water 1L,Beverages,6,4
2,SKU003,Juice 1L,Beverages,6,2
3,SKU004,Energy Drink 250ml,Beverages,3,4
4,SKU005,Iced Tea 500ml,Beverages,6,3


In [50]:
window = 7
horizon = 14

daily_rows =[]
for product_id, row in df_sales.groupby('product_id'):
    units = row['units_sold'].to_numpy(dtype=float)
    last_date = row['date'].max()

    if len(units) < window:
        continue
    avg = units[-window:].mean()
    future_dates = pd.date_range(last_date + pd.Timedelta(days=1), periods = horizon)

    
    for d in future_dates:
        daily_rows.append({
            'product_id':product_id,
            'forecast_dates':d.date(),
            'forecast_units_sold': round(avg,1)
        })

daily = pd.DataFrame(daily_rows)
daily
   
    




,product_id,forecast_dates,forecast_units_sold
0,SKU001,2026-01-01,49.1
1,SKU001,2026-01-02,49.1
2,SKU001,2026-01-03,49.1
3,SKU001,2026-01-04,49.1
4,SKU001,2026-01-05,49.1
...,...,...,...
345,SKU025,2026-01-10,73.0
346,SKU025,2026-01-11,73.0
347,SKU025,2026-01-12,73.0
348,SKU025,2026-01-13,73.0


In [51]:
forecasted_table = pd.merge(daily,df_products, on='product_id', how = 'left')
forecasted_table

,product_id,forecast_dates,forecast_units_sold,product_name,category,lead_time_days,safety_stock_days
0,SKU001,2026-01-01,49.1,Soda 500ml,Beverages,5,4
1,SKU001,2026-01-02,49.1,Soda 500ml,Beverages,5,4
2,SKU001,2026-01-03,49.1,Soda 500ml,Beverages,5,4
3,SKU001,2026-01-04,49.1,Soda 500ml,Beverages,5,4
4,SKU001,2026-01-05,49.1,Soda 500ml,Beverages,5,4
...,...,...,...,...,...,...,...
345,SKU025,2026-01-10,73.0,Sukuma Wiki Bunch,Produce,8,2
346,SKU025,2026-01-11,73.0,Sukuma Wiki Bunch,Produce,8,2
347,SKU025,2026-01-12,73.0,Sukuma Wiki Bunch,Produce,8,2
348,SKU025,2026-01-13,73.0,Sukuma Wiki Bunch,Produce,8,2


`cumcount()` is used for numbering in pandas , just like `row_number()` or `rank()` in window functions SQL. numbering starts at 0.The +1 tells pandas to start numbering from 1

In [52]:
forecasted_table["rank"] = daily.groupby("product_id").cumcount() + 1 
forecasted_table

,product_id,forecast_dates,forecast_units_sold,product_name,category,lead_time_days,safety_stock_days,rank
0,SKU001,2026-01-01,49.1,Soda 500ml,Beverages,5,4,1
1,SKU001,2026-01-02,49.1,Soda 500ml,Beverages,5,4,2
2,SKU001,2026-01-03,49.1,Soda 500ml,Beverages,5,4,3
3,SKU001,2026-01-04,49.1,Soda 500ml,Beverages,5,4,4
4,SKU001,2026-01-05,49.1,Soda 500ml,Beverages,5,4,5
...,...,...,...,...,...,...,...,...
345,SKU025,2026-01-10,73.0,Sukuma Wiki Bunch,Produce,8,2,10
346,SKU025,2026-01-11,73.0,Sukuma Wiki Bunch,Produce,8,2,11
347,SKU025,2026-01-12,73.0,Sukuma Wiki Bunch,Produce,8,2,12
348,SKU025,2026-01-13,73.0,Sukuma Wiki Bunch,Produce,8,2,13


In [53]:
forecasted_table.to_csv(r"C:\Users\ahmed\Documents\Data Science LUXDEV\PPORTFOLIO PROJECTS\INVENTORY FORECASTING\CSV\CLEAN\14_day_forecast.csv", index=False)

In [54]:
horizon = 14

def summarise(per_sku):
    name = per_sku['product_name'].iloc[0]
    category = per_sku['category'].iloc[0]
    total_forecast = per_sku['forecast_units_sold'].sum()
    lead_days =int( per_sku['lead_time_days'].iloc[0])
    safety_days = int(per_sku['safety_stock_days'].iloc[0])
    daily_average = per_sku['forecast_units_sold'].mean()

    in_bracket = per_sku.loc[per_sku["rank"] <= lead_days, "forecast_units_sold"].sum()
    lead_time_demand = in_bracket if lead_days <= horizon else daily_average * lead_days

    safety_units = safety_days * daily_average
    row = pd.Series({
        'product_name':name,
        'product_category':category,
        'average_daily_forecast': round(daily_average,1),
        'lead_time_days':lead_days,
        'safety_stock_days':safety_days,
        'leadtime_demand':round(lead_time_demand,1),
        'safetystock_units': round(safety_units,1),
        'reorder_point': round((safety_units + lead_time_demand),1),
        'forecast_14_day_total':round(total_forecast,1)
    })
    return row

summary = forecasted_table.groupby("product_id").apply(summarise).reset_index()
summary

C:\Users\ahmed\AppData\Local\Temp\ipykernel_11804\227407276.py:28: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  summary = forecasted_table.groupby("product_id").apply(summarise).reset_index()


,product_id,product_name,product_category,average_daily_forecast,lead_time_days,safety_stock_days,leadtime_demand,safetystock_units,reorder_point,forecast_14_day_total
0,SKU001,Soda 500ml,Beverages,49.1,5,4,245.5,196.4,441.9,687.4
1,SKU002,Bottled Water 1L,Beverages,50.4,6,4,302.4,201.6,504.0,705.6
2,SKU003,Juice 1L,Beverages,47.4,6,2,284.4,94.8,379.2,663.6
3,SKU004,Energy Drink 250ml,Beverages,36.7,3,4,110.1,146.8,256.9,513.8
4,SKU005,Iced Tea 500ml,Beverages,66.3,6,3,397.8,198.9,596.7,928.2
5,SKU006,Potato Crisps,Snacks,37.3,9,2,335.7,74.6,410.3,522.2
6,SKU007,Chocolate Bar,Snacks,53.4,3,4,160.2,213.6,373.8,747.6
7,SKU008,Biscuits Pack,Snacks,63.7,5,4,318.5,254.8,573.3,891.8
8,SKU009,Peanuts 200g,Snacks,74.3,4,3,297.2,222.9,520.1,1040.2
9,SKU010,Popcorn Bag,Snacks,37.4,7,4,261.8,149.6,411.4,523.6


In [55]:
summary.to_csv(r"C:\Users\ahmed\Documents\Data Science LUXDEV\PPORTFOLIO PROJECTS\INVENTORY FORECASTING\CSV\CLEAN\forecast_summary.csv",index = False)